**RNNs from scratch:**

In [1]:
import torch
import torch.nn as nn

In [2]:
torch.manual_seed(1)

rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

In [3]:
w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0

In [4]:
print('W_xh shape: ', w_xh.shape)
print('W_hh shape: ', w_hh.shape)
print('b_xh shape: ', b_xh.shape)
print('b_hh shape: ', b_hh.shape)

W_xh shape:  torch.Size([2, 5])
W_hh shape:  torch.Size([2, 2])
b_xh shape:  torch.Size([2])
b_hh shape:  torch.Size([2])


The input shape for this layer is (batch_size, seq_length, 5)...first dim is the batch size, second dim is the sequence length and last dim are the features

In [5]:
x_seq = torch.tensor([[1.0]*5, [2.0]*5, [3.0]*5]).float()

In [6]:
x_seq.shape

torch.Size([3, 5])

In [7]:
x_seq = torch.reshape(x_seq, (1,3,5))

In [8]:
output, hn = rnn_layer(x_seq)

In [9]:
output.shape

torch.Size([1, 3, 2])

In [10]:
hn.shape

torch.Size([1, 1, 2])

In [11]:
print(hn)

tensor([[[-0.8649,  0.9047]]], grad_fn=<StackBackward0>)


In [13]:
# manually computing the output
# the rnn from scratch

out_man = []

# looping through the sequence / unrolling the sequence through time
for t in range(3):
    xt = x_seq[:, t, :]
    print(f"Time step {t} =>")
    print(" Input :", xt.numpy())

    ht = torch.matmul(xt, torch.transpose(w_xh, 0, 1)) + b_xh
    print(' Hidden :', ht.detach().numpy())
    if t > 0:
        prev_h = out_man[t-1]
    else:
        prev_h = torch.zeros((ht.shape))
    
    ot = ht + torch.matmul(prev_h, torch.transpose(w_hh, 0, 1)) + b_hh
    ot = torch.tanh(ot)
    out_man.append(ot)
    print(' Output (manual): ', ot.detach().numpy())
    print(' RNN output: ', output[:, t].detach().numpy())
    print()


Time step 0 =>
 Input : [[1. 1. 1. 1. 1.]]
 Hidden : [[-0.47019297  0.58639044]]
 Output (manual):  [[-0.35198015  0.52525216]]
 RNN output:  [[-0.3519801   0.52525216]]

Time step 1 =>
 Input : [[2. 2. 2. 2. 2.]]
 Hidden : [[-0.8888316  1.2364398]]
 Output (manual):  [[-0.68424344  0.76074266]]
 RNN output:  [[-0.68424344  0.76074266]]

Time step 2 =>
 Input : [[3. 3. 3. 3. 3.]]
 Hidden : [[-1.3074702  1.8864892]]
 Output (manual):  [[-0.8649416  0.9046636]]
 RNN output:  [[-0.8649416  0.9046636]]



In [28]:
out_man

[tensor([[-0.3520,  0.5253]], grad_fn=<TanhBackward0>),
 tensor([[-0.6842,  0.7607]], grad_fn=<TanhBackward0>),
 tensor([[-0.8649,  0.9047]], grad_fn=<TanhBackward0>)]

In [37]:
torch.stack(out_man, dim=1)

tensor([[[-0.3520,  0.5253],
         [-0.6842,  0.7607],
         [-0.8649,  0.9047]]], grad_fn=<StackBackward0>)

In [34]:
out_man[0].shape

torch.Size([1, 2])

In [31]:
output

tensor([[[-0.3520,  0.5253],
         [-0.6842,  0.7607],
         [-0.8649,  0.9047]]], grad_fn=<TransposeBackward1>)

In [35]:
output.shape

torch.Size([1, 3, 2])

Now, implement a 2 layer rnn from scratch

In [15]:
torch.manual_seed(1)

rnn_layer2 = nn.RNN(input_size=5, hidden_size=2, num_layers=2, batch_first=True)

In [18]:
w_xh1, w_xh2 = rnn_layer2.weight_ih_l0, rnn_layer2.weight_ih_l1
w_hh1, w_hh2 = rnn_layer2.weight_hh_l0, rnn_layer2.weight_hh_l1
b_xh1, b_xh2 = rnn_layer2.bias_ih_l0, rnn_layer2.bias_ih_l1
b_hh1, b_hh2 = rnn_layer2.bias_hh_l0, rnn_layer2.bias_hh_l1

In [20]:
output2, hn2 = rnn_layer2(x_seq)

In [21]:
output2

tensor([[[-0.3555,  0.5987],
         [-0.5822,  0.6520],
         [-0.6262,  0.6740]]], grad_fn=<TransposeBackward1>)

In [22]:
output2.shape

torch.Size([1, 3, 2])

In [23]:
hn2

tensor([[[-0.8649,  0.9047]],

        [[-0.6262,  0.6740]]], grad_fn=<StackBackward0>)

In [24]:
hn2.shape

torch.Size([2, 1, 2])

In [25]:
x_seq[:, 0].shape

torch.Size([1, 5])

In [26]:
w_xh1.shape

torch.Size([2, 5])

In [27]:
x_seq[:, 0] @ w_xh1.transpose(1,0)

tensor([[-0.4186,  0.6500]], grad_fn=<MmBackward0>)

Now building the 2 layer RNN from scratch

In [56]:
# iterate through the layers
prev_h_l0 = []
prev_h_l1 = []

layers = 2
timesteps = 3
for i in range(layers):
    # for each layer, the sequence length remains as 3
    # unrolling through time
    for t in range(timesteps):
        # only difference is that the input changes for each layer
        # layer 1 has x_seq as input and layer 2 has the output of layer 1 for an input
        if t > 0:
            prev_h = prev_h_l0[t-1] if i==0 else prev_h_l1[t-1]
        elif t==0:
            prev_h = torch.zeros((ht.shape))


        if i == 0:
            ht = x_seq[:, t] @ w_xh1.transpose(1,0) + b_xh1
            ot = ht + prev_h @ w_hh1.transpose(1,0) + b_hh1
            ot = torch.tanh(ot)
            prev_h_l0.append(ot)

        else:
            ht = prev_h_l0[:, t] @ w_xh2.transpose(1,0) + b_xh2
            ot = ht + prev_h @ w_hh2.transpose(1,0) + b_hh2
            ot = torch.tanh(ot)
            prev_h_l1.append(ot)
        
        if i == 0 and t == 2:
            prev_h_l0 = torch.stack(prev_h_l0, dim=1).detach()  

In [57]:
output2

tensor([[[-0.3555,  0.5987],
         [-0.5822,  0.6520],
         [-0.6262,  0.6740]]], grad_fn=<TransposeBackward1>)

In [58]:
torch.stack(prev_h_l1, dim=1)

tensor([[[-0.3555,  0.5987],
         [-0.5822,  0.6520],
         [-0.6262,  0.6740]]], grad_fn=<StackBackward0>)

identical outputs

In [59]:
hn2

tensor([[[-0.8649,  0.9047]],

        [[-0.6262,  0.6740]]], grad_fn=<StackBackward0>)

In [61]:
prev_h_l0[:, -1], prev_h_l1[-1]

(tensor([[-0.8649,  0.9047]]),
 tensor([[-0.6262,  0.6740]], grad_fn=<TanhBackward0>))

In [2]:
import torch

In [3]:
a = torch.randn((4, 256, 64))

In [4]:
a.shape

torch.Size([4, 256, 64])

In [5]:
a1, a2 = a[-1, :, :], a[-2, :, :]

In [6]:
a1.shape

torch.Size([256, 64])

In [7]:
a2.shape

torch.Size([256, 64])

In [9]:
torch.stack([a1, a2], dim=1).shape

torch.Size([256, 2, 64])

In [12]:
torch.concat([a1, a2], dim=1).shape

torch.Size([256, 128])